- 같은 `subject_id`는 반드시 하나의 split에 존재 (subject-level)
- `ever_delirium` 기준으로 stratified random split
- 12시간 bin-level LSTM 입력은 4개 input step과 3개 target step(`t`, `t+1`, `t+2`)을 기준으로 구성


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
TEST_SIZE    = 0.20
INPUT_STEPS  = 4
TARGET_STEPS = 3

SRC_DIR       = Path.cwd().resolve()
PROJECT_DIR   = SRC_DIR.parent
TRANSFORM_DIR = PROJECT_DIR / "processed" / "transform"
MODELING_DIR  = PROJECT_DIR / "processed" / "modeling"

In [ ]:
input_files = {
    "binned": TRANSFORM_DIR / "events_12h_binned.csv",
    "cohort": TRANSFORM_DIR / "cohort_final.csv",
}

events_binned = pd.read_csv(input_files["binned"])
cohort        = pd.read_csv(input_files["cohort"])

print("events_binned:", events_binned.shape)
print("cohort:", cohort.shape)

events_binned["bin"]      = pd.to_numeric(events_binned["bin"], errors="raise").astype(int)
events_binned["delirium"] = pd.to_numeric(events_binned["delirium"], errors="coerce")
cohort["ever_delirium"]   = pd.to_numeric(cohort["ever_delirium"], errors="coerce").fillna(0).astype(int)

## 환자 단위 train/test 분할

In [ ]:
def make_subject_summary(cohort_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        cohort_df.groupby("subject_id", as_index=False)
        .agg(
            ever_delirium=("ever_delirium", "max"),
            n_stays=("stay_id", "nunique"),
        )
        .sort_values("subject_id")
        .reset_index(drop=True)
    )
    summary["ever_delirium"] = summary["ever_delirium"].fillna(0).astype(int)
    return summary

def subject_level_split(
    subject_summary: pd.DataFrame,
    test_size: float = 0.2,
    random_state: int = 42,
    stratify_col: str = "ever_delirium",
) -> tuple[set, set, str]:
    rng = np.random.default_rng(random_state)
    subjects = subject_summary["subject_id"].to_numpy()
    labels = subject_summary[stratify_col]
    class_counts = labels.value_counts(dropna=False)

    can_stratify = labels.nunique(dropna=False) > 1 and class_counts.min() >= 2
    test_subjects = []

    if can_stratify:
        split_method = f"stratified_by_{stratify_col}"
        for _, group in subject_summary.groupby(stratify_col, dropna=False):
            group_subjects = group["subject_id"].to_numpy()
            n_test = int(round(len(group_subjects) * test_size))
            n_test = min(max(n_test, 1), len(group_subjects) - 1)
            test_subjects.extend(rng.choice(group_subjects, size=n_test, replace=False).tolist())
    else:
        split_method = "random_subject_split"
        n_test = int(round(len(subjects) * test_size))
        n_test = min(max(n_test, 1), len(subjects) - 1)
        test_subjects = rng.choice(subjects, size=n_test, replace=False).tolist()

    test_subjects = set(test_subjects)
    train_subjects = set(subjects) - test_subjects
    return train_subjects, test_subjects, split_method

In [ ]:
subject_summary = make_subject_summary(cohort)
train_subjects, test_subjects, split_method = subject_level_split(
    subject_summary,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify_col="ever_delirium",
)

subject_summary["split"] = np.where(subject_summary["subject_id"].isin(train_subjects), "train", "test")

split_label_summary = (
    pd.crosstab(subject_summary["split"], subject_summary["ever_delirium"])
    .reindex(index=["train", "test"], fill_value=0)
    .rename(columns={0: "ever_delirium_0", 1: "ever_delirium_1"})
)
for col in ["ever_delirium_0", "ever_delirium_1"]:
    if col not in split_label_summary.columns:
        split_label_summary[col] = 0
split_label_summary = split_label_summary[["ever_delirium_0", "ever_delirium_1"]]
split_label_summary["n_subjects"] = split_label_summary.sum(axis=1)

display(split_label_summary)


## split 컬럼 부여

In [ ]:
split_map = subject_summary.set_index("subject_id")["split"]

def attach_split(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out["split"] = out["subject_id"].map(split_map)
    return out

events_binned_split = attach_split(events_binned, "events_binned")
cohort_split = attach_split(cohort, "cohort")

print(events_binned_split["split"].value_counts())
print(cohort_split["split"].value_counts())

## LSTM sequence index 생성

In [58]:
def _format_bin_tokens(tokens: list) -> str:
    # PAD 위치의 명시적 토큰 저장, downstream tensor 생성 시 zero row 변환용
    return ",".join("PAD" if token is None else str(int(token)) for token in tokens)


def build_lstm_sequence_index(
    binned_df: pd.DataFrame,
    input_steps: int = 4,
    target_steps: int = 3,
) -> pd.DataFrame:
    # ICU stay 안에서 anchor를 오른쪽으로 sliding하며 sequence row 생성
    records = []

    for stay_id, stay_df in binned_df.groupby("stay_id", sort=False):
        # 12시간 binned table의 bin당 1개 row 보장과 정렬 결정성 유지
        stay_df = stay_df.drop_duplicates("bin", keep="first").sort_values("bin").reset_index(drop=True)
        bins = stay_df["bin"].to_numpy(dtype=int)
        labels = pd.to_numeric(stay_df["delirium"], errors="coerce").to_numpy()
        n_bins = len(stay_df)

        # 24시간 이상 stay만 남긴 구조라 첫 anchor는 bin 2, 최소 입력은 PAD,PAD,1,2
        for anchor_pos in range(1, n_bins):
            # X는 anchor까지의 최근 4개 bin 사용, 부족한 과거 step은 left padding
            input_pos = list(range(max(0, anchor_pos - input_steps + 1), anchor_pos + 1))
            input_bins = [None] * (input_steps - len(input_pos)) + bins[input_pos].tolist()
            input_mask = [0] * (input_steps - len(input_pos)) + [1] * len(input_pos)

            # Y는 anchor bin에서 시작, y_t, y_t_plus_1, y_t_plus_2 구성
            target_pos = list(range(anchor_pos, min(n_bins, anchor_pos + target_steps)))
            target_bins = bins[target_pos].tolist() + [None] * (target_steps - len(target_pos))
            target_labels = [int(label) for label in labels[target_pos].tolist()] + [0] * (target_steps - len(target_pos))
            # Target PAD 위치만 mask 0으로 loss/metric 제외
            target_mask = [1] * len(target_pos) + [0] * (target_steps - len(target_pos))

            anchor = stay_df.iloc[anchor_pos]
            example_id = f"{int(stay_id)}_anchor{int(bins[anchor_pos])}"
            # preprocessing/modeling의 암묵적 padding 추론 방지를 위한 bin token과 mask 동시 저장
            record = {
                "example_id": example_id,
                "subject_id": anchor["subject_id"],
                "hadm_id": anchor["hadm_id"],
                "stay_id": stay_id,
                "split": anchor["split"],
                "n_observed_bins": n_bins,
                "anchor_bin": int(bins[anchor_pos]),
                "input_start_bin": int(bins[input_pos[0]]),
                "input_end_bin": int(bins[input_pos[-1]]),
                "target_start_bin": int(bins[target_pos[0]]),
                "target_end_bin": int(bins[target_pos[-1]]),
                "input_bins": _format_bin_tokens(input_bins),
                "input_mask": ",".join(map(str, input_mask)),
                "target_bins": _format_bin_tokens(target_bins),
                "target_mask": ",".join(map(str, target_mask)),
                "target_available_count": int(sum(target_mask)),
            }
            # masked BCE 학습용 horizon별 label과 mask 병렬 저장
            for offset, (label, mask) in enumerate(zip(target_labels, target_mask)):
                label_col = "y_t" if offset == 0 else f"y_t_plus_{offset}"
                record[label_col] = label
                record[f"{label_col}_mask"] = mask
            # masked loss와 동일하게 padded horizon을 제외한 summary target 계산
            record["y_any_t_to_t_plus_2"] = int(any(label == 1 and mask == 1 for label, mask in zip(target_labels, target_mask)))
            records.append(record)

    sequence_index = pd.DataFrame(records)
    return sequence_index


In [59]:
sequence_index = build_lstm_sequence_index(
    events_binned_split,
    input_steps=INPUT_STEPS,
    target_steps=TARGET_STEPS,
)

sequence_train = sequence_index[sequence_index["split"] == "train"].reset_index(drop=True)
sequence_test  = sequence_index[sequence_index["split"] == "test"].reset_index(drop=True)

print("sequence_index:", sequence_index.shape)
display(sequence_index["split"].value_counts())


sequence_index: (6878, 23)


split
train    4934
test     1944
Name: count, dtype: int64

## 산출물 저장

In [62]:
train_subject_ids = pd.DataFrame({"subject_id": sorted(train_subjects)})
test_subject_ids = pd.DataFrame({"subject_id": sorted(test_subjects)})

split_summary = pd.concat(
    [
        sequence_index.groupby("split", as_index=False)
        .agg(
            n_rows=("example_id", "count"),
            n_subjects=("subject_id", "nunique"),
            n_stays=("stay_id", "nunique"),
            y_positive=("y_any_t_to_t_plus_2", "sum"),
            y_positive_rate=("y_any_t_to_t_plus_2", "mean"),
        )
        .assign(table="lstm_sequence_index"),
    ],
    ignore_index=True,
    sort=False,
)

events_binned_split.to_csv(MODELING_DIR / "events_12h_binned_with_split.csv", index=False)
cohort_split.to_csv(MODELING_DIR / "cohort_final_with_split.csv", index=False)

train_subject_ids.to_csv(MODELING_DIR / "train_subject_ids.csv", index=False)
test_subject_ids.to_csv(MODELING_DIR / "test_subject_ids.csv", index=False)
subject_summary.to_csv(MODELING_DIR / "subject_split_summary.csv", index=False)
split_summary.to_csv(MODELING_DIR / "train_test_split_summary.csv", index=False)

sequence_index.to_csv(MODELING_DIR / "lstm_sequence_index.csv", index=False)
sequence_train.to_csv(MODELING_DIR / "lstm_sequence_index_train.csv", index=False)
sequence_test.to_csv(MODELING_DIR / "lstm_sequence_index_test.csv", index=False)

print("Saved outputs to", MODELING_DIR)
display(split_summary)

Saved outputs to D:\ONEASH_local\Delirium-Prediction\Parkinson\processed\modeling


,split,n_rows,n_subjects,n_stays,y_positive,y_positive_rate,table
0,test,1944,98,177,657,0.337963,lstm_sequence_index
1,train,4934,391,598,1611,0.326510,lstm_sequence_index
